In [1]:
import sys; sys.path.append("..")
import time, numpy as np, pandas as pd
from transformers import pipeline
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             precision_recall_curve, average_precision_score)
from src.data import load_clean
from src.utils import log_result

train, val, test, _ = load_clean()
y = val["sarcasm"].values
texts = val["clean"].tolist()          # texte sans artefact, emojis/ponctuation conservés

def best_threshold(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    f1s = 2 * prec * rec / (prec + rec + 1e-12)
    i = f1s[:-1].argmax()
    return thr[i], f1s[i]

def report(name, y, p):
    t, f1 = best_threshold(y, p)
    print(f"{name}: seuil={t:.3f} | F1={f1:.4f} | PR-AUC={average_precision_score(y, p):.4f} "
          f"| macro-F1={f1_score(y, p >= t, average='macro'):.4f}")
    return t, f1

retirés (présents dans test): 4 | (présents dans val): 72 | groupes contradictoires: 47 | doublons restants retirés: 287
tailles -> train: 10834, val: 1255, test: 3000


In [2]:
sa = pipeline("text-classification",
              model="CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment",
              top_k=None, truncation=True, max_length=128, device=-1)
print(sa.model.config.id2label)          # à noter pour le compte-rendu

t0 = time.time()
out = sa(texts, batch_size=32)
dur_b2 = time.time() - t0
print("durée (s):", round(dur_b2, 1))

def p_label(o, prefix):
    return next(d["score"] for d in o if d["label"].lower().startswith(prefix))

pneg = np.array([p_label(o, "neg") for o in out])
pred_hard = np.array([max(o, key=lambda d: d["score"])["label"].lower().startswith("neg")
                      for o in out])

print("\n--- Règle dure : négatif => sarcastique ---")
print("précision:", round(precision_score(y, pred_hard), 3),
      "| rappel:", round(recall_score(y, pred_hard), 3),
      "| F1:", round(f1_score(y, pred_hard), 3))

t_b2, f1_b2 = report("P(négatif) seuillé sur val", y, pneg)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{0: 'positive', 1: 'negative', 2: 'neutral'}
durée (s): 166.1

--- Règle dure : négatif => sarcastique ---
précision: 0.332 | rappel: 0.797 | F1: 0.469
P(négatif) seuillé sur val: seuil=0.949 | F1=0.4929 | PR-AUC=0.4735 | macro-F1=0.6842


In [3]:
log_result(dict(model="camelbert_sent_neg_rule", preprocessing="clean", seed=0,
                threshold=round(t_b2, 4), f1_val=round(f1_b2, 4),
                prauc_val=round(average_precision_score(y, pneg), 4),
                macrof1_val=round(f1_score(y, pneg >= t_b2, average="macro"), 4),
                f1_test="", prauc_test="", macrof1_test="",
                train_time_s=0, trained_params=0),
           path="../results/results.csv")
print("fin")

fin


In [ ]:
zs = pipeline("zero-shot-classification",
              model="MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7",
              device=-1)

formulations = {
    "arabe":    (["ساخرة", "جادة"],         "هذه التغريدة {}."),
    "français": (["sarcastique", "sérieux"], "Ce tweet est {}."),
    "anglais":  (["sarcastic", "serious"],   "This tweet is {}."),
}

def score_zs(texts, labels, template, bs=16):
    res = zs(texts, candidate_labels=labels, hypothesis_template=template, batch_size=bs)
    return np.array([r["scores"][r["labels"].index(labels[0])] for r in res])

# Essai rapide : 200 premiers tweets de val
n = 200
t0 = time.time()
p_try = score_zs(texts[:n], *formulations["anglais"])
print("durée pour 200 tweets (s):", round(time.time() - t0, 1))
print("F1 (essai, seuil optimisé sur ces 200):", round(best_threshold(y[:n], p_try)[1], 3))

C:\Users\oussa\tp2-sarcasme\.venv\lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [ ]:
scores_b1 = {}
for name, (labels, tpl) in formulations.items():
    t0 = time.time()
    p = score_zs(texts, labels, tpl)
    scores_b1[name] = p
    t, f1 = report(f"B1 {name}", y, p)
    log_result(dict(model=f"mdeberta_nli_zs_{name}", preprocessing="clean", seed=0,
                    threshold=round(t, 4), f1_val=round(f1, 4),
                    prauc_val=round(average_precision_score(y, p), 4),
                    macrof1_val=round(f1_score(y, p >= t, average="macro"), 4),
                    f1_test="", prauc_test="", macrof1_test="",
                    train_time_s=0, trained_params=0),
               path="../results/results.csv")
    print("   durée (s):", round(time.time() - t0, 1))